# DD-PRiSM-plus — Step 1: set up and fetch all data

Run this once in a **CPU** session (no GPU — preprocessing needs none, and
GPU quota is only ~30 h/week). At the end, click **Save Version** so the
downloaded data persists as a reusable dataset.

**Before running:** open the right-hand panel (`<` arrow, top right) and set
- **Accelerator → None**
- **Internet → On**  ← without this nothing downloads

In [ ]:
# 0. Confirm the session is set up correctly
import subprocess, sys, os

try:
    import torch
    print(f'torch {torch.__version__}  cuda_available={torch.cuda.is_available()}')
except ImportError:
    print('torch not present (fine for this notebook)')

ok = subprocess.run(['curl','-sI','--max-time','15','https://github.com'],
                    capture_output=True).returncode == 0
print('internet:', 'ON' if ok else 'OFF  <-- enable it in Session options, then rerun')
print('free disk:', subprocess.run(['df','-h','/kaggle/working'],
      capture_output=True, text=True).stdout.splitlines()[-1])

## 1. Get the code

In [ ]:
REPO = '/kaggle/working/ddprism-plus'

if os.path.exists(REPO):
    print('already cloned; pulling latest')
    !cd {REPO} && git pull --quiet
else:
    !git clone --quiet https://github.com/SanaNiroomand/DD-PRiSM-plus.git {REPO}

os.chdir(REPO)
print('working in', os.getcwd())

## 2. Install the two packages Kaggle lacks

`zipfile-deflate64` is **mandatory** — DOSERESP.zip uses Deflate64 and the
standard library cannot decompress it. `rdkit` is only needed if you derive
SMILES from Chem2D rather than using `nsc_smiles.csv`.

In [ ]:
!pip install --quiet zipfile-deflate64 rdkit openpyxl
print('installed')

## 3. Sanity-check the model code

23 tests, a few seconds. They prove the vectorised model still matches the
published one exactly. If these fail, stop — do not train.

In [ ]:
!python -m pytest tests -q

## 4. Download every source dataset (~1 GB)

All eight files, straight onto Kaggle. Nothing is uploaded from your laptop.

If a DepMap file returns **202 Accepted**, figshare is throttling this
address — rerun the cell later with `--only depmap_expression --attempts 9`.

In [ ]:
DATA = '/kaggle/working/data'
!python scripts/get_data.py --dest {DATA} --include-optional --attempts 9

## 5. Verify before trusting any of it

Every required row must read `ok`. This check has caught four different
silent failures (truncation, empty bodies, throttle responses), so do not
skip it.

In [ ]:
!python scripts/get_data.py --dest {DATA} --check

In [ ]:
# What actually landed
!ls -la {DATA}/Raw && du -sh {DATA}

## 6. Save it

Click **Save Version** (top right) → *Save & Run All (Commit)*.

That freezes `/kaggle/working` into a versioned output. In the next notebook,
add it via **Add Input → Your Work → Notebook Output**, and it appears
read-only under `/kaggle/input/…` — which does **not** count against the
20 GB working quota.

Without this step everything here is deleted when the session ends.

---

**Next:** preprocessing. The target is exactly **7,915,900** NCI60 training
rows and **1,387,317** combination rows — those two numbers are how you know
it worked.